In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker
import glasbey
import pickle
import time
from pathlib import Path
import distro

from tqdm.notebook import tqdm
from collections import defaultdict

# for sanity check
from transformers import AutoTokenizer, AutoModel
from transformers import BertConfig, BertModel
from transformers import RobertaTokenizer, RobertaModel
import mteb
from sentence_transformers import SentenceTransformer, models

%load_ext watermark

2026-02-20 16:16:40.818262: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-20 16:16:40.830962: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771600600.842807  448382 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771600600.846454  448382 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-20 16:16:40.862434: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [ ]:
# old one '1.8.1+cu111'
torch.__version__

'2.5.0+cu124'

In [ ]:
%load_ext autoreload
%autoreload 2

from text_embeddings_src.train_stuff import *
from text_embeddings_src.eval_functions import KNNEval, MTEBEval
from text_embeddings_src.models import HFModelWrapper, FineTunedHFModelWrapper
from text_embeddings_src.data_stuff import (
    MultOverlappingSentencesPairDataset,
)

In [ ]:
import black
import jupyter_black

jupyter_black.load(line_length=79)

In [ ]:
variables_path = Path("../results/variables")
figures_path = Path("../results/figures/updated_dataset")
data_path = Path("../data")

In [ ]:
# MANUAL FIX TO PATH ISSUE FROM VSCODE
import text_embeddings_src

nb_path = Path(text_embeddings_src.__path__[0]).parents[0] / Path("scripts")
assert nb_path.exists(), "The path does not exist"

variables_path = (nb_path / variables_path).resolve(strict=True)
figures_path = (nb_path / figures_path).resolve(strict=True)
data_path = (nb_path / data_path).resolve(strict=True)

In [ ]:
plt.style.use((nb_path / Path("matplotlib_style.txt")).resolve(strict=True))

In [ ]:
%watermark -a 'Rita González-Márquez' -t -d -tz -u -v -iv -w -m -h
print(distro.name(pretty=True))

Author: Rita González-Márquez

Last updated: 2026-02-20 16:17:40CET

Python implementation: CPython
Python version       : 3.12.4
IPython version      : 8.31.0

Compiler    : GCC 11.2.0
OS          : Linux
Release     : 5.14.0-570.17.1.el9_6.x86_64
Machine     : x86_64
Processor   : x86_64
CPU cores   : 64
Architecture: 64bit

Hostname: rgonzalesmarquez_GPU0-llm_gber7

matplotlib           : 3.9.2
torch                : 2.5.0
distro               : 1.9.0
transformers         : 4.45.2
text_embeddings_src  : 0.0.0
pandas               : 2.2.3
black                : 24.10.0
jupyter_black        : 0.4.0
numpy                : 1.26.4
glasbey              : 0.2.1
tqdm                 : 4.66.4
mteb                 : 1.19.9
sentence_transformers: 3.3.0

Watermark: 2.5.0

Ubuntu 24.04 LTS


# Evaluation after every layer

## Pre-trained MPNet

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)

# wrap model
wrapped_model = HFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda


Some weights of MPNetModel were not initialized from the model checkpoint at microsoft/mpnet-base and are newly initialized: ['mpnet.pooler.dense.bias', 'mpnet.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPU times: user 192 ms, sys: 175 ms, total: 367 ms
Wall time: 3.66 s


In [ ]:
tasks = [
    "ArxivClusteringP2P",
    "BiorxivClusteringP2P",
    "MedrxivClusteringP2P",
    "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    "SciDocsRR",
    "MindSmallReranking",
    "SCIDOCS",
    "ArguAna",
    "STS15",
    "STS16",
    "STSBenchmark",
    ## classification
    "AmazonPolarityClassification",
    "Banking77Classification",
    "ImdbClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    "MTOPDomainClassification",
    "TweetSentimentExtractionClassification",
]

In [ ]:
# eval
eval_results = defaultdict(list)

for layer_number in np.arange(12):#(13):
# layer_number = 12
    print("layer number", layer_number)
    eval_results["layer"].append(layer_number)

    saving_path = (
        Path("embeddings_" + model_name.lower())
        / Path("updated_dataset")
        / Path("mteb_benchmark")
        / Path("eval_layers")
        / Path(f"results_{model_name.lower()}")
    )
    mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
    (mteb_save_path).mkdir(exist_ok=True, parents=True)

    dict_results = MTEBEval(
        wrapped_model=wrapped_model,
        tasks=tasks,
        path_to_save=mteb_save_path,
        eval_rep="av",
        layer_number=layer_number,
    )
    [eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number 0


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

- Banking77Classification, s2s

...

STS

- STS15, s2s

- STS16, s2s

- STSBenchmark, s2s

Clustering:  74%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                            | 23/31 [1:18:18<23:12, 174.06s/it]

In [ ]:
df_eval_results

,layer,ArxivClusteringP2P,BiorxivClusteringP2P,MedrxivClusteringP2P,RedditClusteringP2P,StackExchangeClusteringP2P,SciDocsRR,MindSmallReranking,SCIDOCS,ArguAna,STS15,STS16,STSBenchmark,AmazonPolarityClassification,Banking77Classification,ImdbClassification,MassiveIntentClassification,MassiveScenarioClassification,MTOPDomainClassification,TweetSentimentExtractionClassification
0,12,0.277987,0.231671,0.22471,0.373795,0.263539,0.560579,0.274671,0.01393,0.22227,0.534896,0.50586,0.51988,0.664892,0.574091,0.61806,0.232448,0.252589,0.759439,0.5279


In [ ]:
model_name = "MPNet"
saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}")
)
df_eval_results = pd.read_parquet(
    variables_path / saving_path / "df_eval_results",
    engine="pyarrow",
)
df_eval_results

,layer,ArxivClusteringP2P,BiorxivClusteringP2P,MedrxivClusteringP2P,RedditClusteringP2P,StackExchangeClusteringP2P,SciDocsRR,MindSmallReranking,SCIDOCS,ArguAna,STS15,STS16,STSBenchmark,AmazonPolarityClassification,Banking77Classification,ImdbClassification,MassiveIntentClassification,MassiveScenarioClassification,MTOPDomainClassification,TweetSentimentExtractionClassification
0,0,0.238123,0.218026,0.210413,0.273557,0.229795,0.676749,0.270333,0.05748,0.34075,0.742692,0.686107,0.633244,0.622958,0.699643,0.552372,0.468796,0.492401,0.860762,0.498896
1,1,0.260717,0.253289,0.232454,0.341346,0.247153,0.666582,0.293889,0.02431,0.33148,0.721731,0.655231,0.583329,0.648317,0.671104,0.578176,0.419973,0.432482,0.849088,0.517883
2,2,0.268973,0.253067,0.236923,0.368180,0.251976,0.630935,0.283086,0.01344,0.28885,0.674465,0.652893,0.554007,0.653211,0.653604,0.583332,0.382112,0.396369,0.837346,0.517629
3,3,0.264569,0.242690,0.230613,0.364578,0.249242,0.627101,0.281777,0.01257,0.25558,0.683329,0.646372,0.561185,0.674060,0.660000,0.594804,0.380935,0.388130,0.827930,0.523939
4,4,0.260083,0.232044,0.222035,0.354656,0.245397,0.615694,0.280419,0.01158,0.24072,0.676184,0.636366,0.563437,0.697055,0.664935,0.608764,0.375757,0.383154,0.819061,0.534720
5,5,0.251286,0.222993,0.215534,0.333902,0.241370,0.601129,0.283478,0.00822,0.23804,0.665067,0.620642,0.564646,0.708475,0.655195,0.627736,0.357835,0.367216,0.807889,0.543152
6,6,0.245970,0.213241,0.209221,0.346821,0.238081,0.585093,0.280049,0.00675,0.21325,0.637176,0.614565,0.549330,0.762176,0.644123,0.687012,0.343443,0.347445,0.793730,0.551245
7,7,0.242547,0.207542,0.201846,0.349362,0.234907,0.567964,0.276525,0.00696,0.19181,0.602605,0.609681,0.523514,0.776562,0.628019,0.704156,0.330935,0.334869,0.778363,0.554924
8,8,0.243564,0.207016,0.207649,0.366650,0.237590,0.578332,0.283198,0.00773,0.20412,0.615948,0.620602,0.538271,0.780894,0.640455,0.700684,0.326496,0.330397,0.791359,0.558263
9,9,0.259655,0.226515,0.216421,0.402686,0.244786,0.584687,0.284972,0.00925,0.22083,0.637528,0.634204,0.557995,0.755956,0.651396,0.690968,0.320444,0.320477,0.806658,0.557187


## Fine-tuned MPNet (Crops)

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "crops"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

# wrap model
wrapped_model = FineTunedHFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda
CPU times: user 196 ms, sys: 88.8 ms, total: 285 ms
Wall time: 2.87 s


In [ ]:
tasks = [
    "ArxivClusteringP2P",
    "BiorxivClusteringP2P",
    "MedrxivClusteringP2P",
    "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    "SciDocsRR",
    "MindSmallReranking",
    "SCIDOCS",
    "ArguAna",
    "STS15",
    "STS16",
    "STSBenchmark",
    ## classification
    "AmazonPolarityClassification",
    "Banking77Classification",
    "ImdbClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    "MTOPDomainClassification",
    "TweetSentimentExtractionClassification",
]

In [ ]:
# eval
eval_results = defaultdict(list)

for layer_number in np.arange(13):
    print("layer number", layer_number)
    eval_results["layer"].append(layer_number)

    saving_path = (
        Path("embeddings_" + model_name.lower())
        / Path("updated_dataset")
        / Path("mteb_benchmark")
        / Path("eval_layers")
        / Path(f"results_{model_name.lower()}_{finetune_type}_finetuning")
    )
    mteb_save_path = (
        variables_path / saving_path / Path(f"layer_{layer_number}")
    )
    (mteb_save_path).mkdir(exist_ok=True, parents=True)

    dict_results = MTEBEval(
        wrapped_model=wrapped_model,
        tasks=tasks,
        path_to_save=mteb_save_path,
        eval_rep="av",
        layer_number=layer_number,
    )
    [eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number 0
Clustering: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [11:08<00:00, 66.80s/it]
Repo card metadata block was not found. Setting CardData to empty.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process 

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

- Banking77Classification, s2s

...

STS

- STS15, s2s

- STS16, s2s

- STSBenchmark, s2s

Clustering:  45%|██████████████████████████████████████████████████████████████████████████████▏                                                                                              | 14/31 [50:17<1:01:24, 216.74s/it]

In [ ]:
dict_results

{'STS15': 0.7247950136316723}

In [ ]:
df_eval_results

,layer,STS15
0,0,0.742643
1,1,0.722556
2,2,0.680495
3,3,0.691847
4,4,0.694313
5,5,0.690845
6,6,0.671847
7,7,0.638158
8,8,0.656962
9,9,0.694827


## Fine-tuned MPNet (Dropout)

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "simcse"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

# wrap model
wrapped_model = FineTunedHFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda
CPU times: user 218 ms, sys: 190 ms, total: 409 ms
Wall time: 2.64 s


In [ ]:
# TODO: change name back to tasks
original_tasks = [
    "ArxivClusteringP2P",
    "BiorxivClusteringP2P",
    "MedrxivClusteringP2P",
    "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    "SciDocsRR",
    "MindSmallReranking",
    "SCIDOCS",
    "ArguAna",
    "STS15",
    "STS16",
    "STSBenchmark",
    ## classification
    "AmazonPolarityClassification",
    "Banking77Classification",
    "ImdbClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    "MTOPDomainClassification",
    "TweetSentimentExtractionClassification",
]

In [ ]:
# eval
eval_results = defaultdict(list)

for layer_number in np.arange(6,13):
    # TODO: delete this whenever finish running
    if layer_number==6:
        tasks = original_tasks[3:]
    else:
        tasks=original_tasks

    print("layer number", layer_number)
    eval_results["layer"].append(layer_number)

    saving_path = (
        Path("embeddings_" + model_name.lower())
        / Path("updated_dataset")
        / Path("mteb_benchmark")
        / Path("eval_layers")
        / Path(f"results_{model_name.lower()}_{finetune_type}_finetuning")
    )
    mteb_save_path = (
        variables_path / saving_path / Path(f"layer_{layer_number}")
    )
    (mteb_save_path).mkdir(exist_ok=True, parents=True)

    dict_results = MTEBEval(
        wrapped_model=wrapped_model,
        tasks=tasks,
        path_to_save=mteb_save_path,
        eval_rep="av",
        layer_number=layer_number,
    )
    [eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number 6


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

- Banking77Classification, s2s

...

STS

- STS15, s2s

- STS16, s2s

- STSBenchmark, s2s

Clustering:   0%|                                                                                                                                                                                         | 0/10 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Clustering:  10%|█████████████████▌                                                                                            

## Pre-trained BERT

In [ ]:
## Next models
- [x] BERT
- [ ] Dropout
- SBERT (/2)

In [ ]:
model_name = "BERT"
model_path = "bert-base-uncased"

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)

# wrap model
wrapped_model = HFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda


Some weights of MPNetModel were not initialized from the model checkpoint at microsoft/mpnet-base and are newly initialized: ['mpnet.pooler.dense.bias', 'mpnet.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPU times: user 108 ms, sys: 12.6 ms, total: 121 ms
Wall time: 3.29 s


In [ ]:
tasks = [
    "ArxivClusteringP2P",
    "BiorxivClusteringP2P",
    "MedrxivClusteringP2P",
    "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    "SciDocsRR",
    "MindSmallReranking",
    "SCIDOCS",
    "ArguAna",
    "STS15",
    "STS16",
    "STSBenchmark",
    ## classification
    "AmazonPolarityClassification",
    "Banking77Classification",
    "ImdbClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    "MTOPDomainClassification",
    "TweetSentimentExtractionClassification",
]

In [ ]:
# eval
eval_results = defaultdict(list)

for layer_number in np.arange(13):
# layer_number = 12
    print("layer number", layer_number)
    eval_results["layer"].append(layer_number)

    saving_path = (
        Path("embeddings_" + model_name.lower())
        / Path("updated_dataset")
        / Path("mteb_benchmark")
        / Path("eval_layers")
        / Path(f"results_{model_name.lower()}")
    )
    mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
    (mteb_save_path).mkdir(exist_ok=True, parents=True)

    dict_results = MTEBEval(
        wrapped_model=wrapped_model,
        tasks=tasks,
        path_to_save=mteb_save_path,
        eval_rep="av",
        layer_number=layer_number,
    )
    [eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number 0
/.pyenv/versions/miniconda3-latest/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after p

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

- Banking77Classification, s2s

...

STS

- STS15, s2s

- STS16, s2s

- STSBenchmark, s2s

Clustering:   0%|                                                                                                                                                                                         | 0/31 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to 

In [ ]:
df_eval_results

,layer,ArxivClusteringP2P,BiorxivClusteringP2P,MedrxivClusteringP2P,RedditClusteringP2P,StackExchangeClusteringP2P,SciDocsRR,MindSmallReranking,SCIDOCS,ArguAna,STS15,STS16,STSBenchmark,AmazonPolarityClassification,Banking77Classification,ImdbClassification,MassiveIntentClassification,MassiveScenarioClassification,MTOPDomainClassification,TweetSentimentExtractionClassification
0,12,0.277987,0.231671,0.22471,0.373795,0.263539,0.560579,0.274671,0.01393,0.22227,0.534896,0.50586,0.51988,0.664892,0.574091,0.61806,0.232448,0.252589,0.759439,0.5279


In [ ]:
model_name = "MPNet"
saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}")
)
df_eval_results = pd.read_parquet(
    variables_path / saving_path / "df_eval_results",
    engine="pyarrow",
)
df_eval_results

,layer,ArxivClusteringP2P,BiorxivClusteringP2P,MedrxivClusteringP2P,RedditClusteringP2P,StackExchangeClusteringP2P,SciDocsRR,MindSmallReranking,SCIDOCS,ArguAna,STS15,STS16,STSBenchmark,AmazonPolarityClassification,Banking77Classification,ImdbClassification,MassiveIntentClassification,MassiveScenarioClassification,MTOPDomainClassification,TweetSentimentExtractionClassification
0,0,0.238123,0.218026,0.210413,0.273557,0.229795,0.676749,0.270333,0.05748,0.34075,0.742692,0.686107,0.633244,0.622958,0.699643,0.552372,0.468796,0.492401,0.860762,0.498896
1,1,0.260717,0.253289,0.232454,0.341346,0.247153,0.666582,0.293889,0.02431,0.33148,0.721731,0.655231,0.583329,0.648317,0.671104,0.578176,0.419973,0.432482,0.849088,0.517883
2,2,0.268973,0.253067,0.236923,0.368180,0.251976,0.630935,0.283086,0.01344,0.28885,0.674465,0.652893,0.554007,0.653211,0.653604,0.583332,0.382112,0.396369,0.837346,0.517629
3,3,0.264569,0.242690,0.230613,0.364578,0.249242,0.627101,0.281777,0.01257,0.25558,0.683329,0.646372,0.561185,0.674060,0.660000,0.594804,0.380935,0.388130,0.827930,0.523939
4,4,0.260083,0.232044,0.222035,0.354656,0.245397,0.615694,0.280419,0.01158,0.24072,0.676184,0.636366,0.563437,0.697055,0.664935,0.608764,0.375757,0.383154,0.819061,0.534720
5,5,0.251286,0.222993,0.215534,0.333902,0.241370,0.601129,0.283478,0.00822,0.23804,0.665067,0.620642,0.564646,0.708475,0.655195,0.627736,0.357835,0.367216,0.807889,0.543152
6,6,0.245970,0.213241,0.209221,0.346821,0.238081,0.585093,0.280049,0.00675,0.21325,0.637176,0.614565,0.549330,0.762176,0.644123,0.687012,0.343443,0.347445,0.793730,0.551245
7,7,0.242547,0.207542,0.201846,0.349362,0.234907,0.567964,0.276525,0.00696,0.19181,0.602605,0.609681,0.523514,0.776562,0.628019,0.704156,0.330935,0.334869,0.778363,0.554924
8,8,0.243564,0.207016,0.207649,0.366650,0.237590,0.578332,0.283198,0.00773,0.20412,0.615948,0.620602,0.538271,0.780894,0.640455,0.700684,0.326496,0.330397,0.791359,0.558263
9,9,0.259655,0.226515,0.216421,0.402686,0.244786,0.584687,0.284972,0.00925,0.22083,0.637528,0.634204,0.557995,0.755956,0.651396,0.690968,0.320444,0.320477,0.806658,0.557187


## Pre-trained SBERT

In [ ]:
model_name = "SBERT"
model_path = "sentence-transformers/all-mpnet-base-v2"

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)

# wrap model
wrapped_model = HFModelWrapper(model, tokenizer)

Model:  SBERT
Running on device: cuda
CPU times: user 183 ms, sys: 172 ms, total: 355 ms
Wall time: 3.5 s


In [ ]:
tasks = [
    "ArxivClusteringP2P",
    "BiorxivClusteringP2P",
    "MedrxivClusteringP2P",
    "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    "SciDocsRR",
    "MindSmallReranking",
    "SCIDOCS",
    "ArguAna",
    "STS15",
    "STS16",
    "STSBenchmark",
    ## classification
    "AmazonPolarityClassification",
    "Banking77Classification",
    "ImdbClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    "MTOPDomainClassification",
    "TweetSentimentExtractionClassification",
]

In [ ]:
# eval
eval_results = defaultdict(list)

for layer_number in np.arange(13):
# layer_number = 12
    print("layer number", layer_number)
    eval_results["layer"].append(layer_number)

    saving_path = (
        Path("embeddings_" + model_name.lower())
        / Path("updated_dataset")
        / Path("mteb_benchmark")
        / Path("eval_layers")
        / Path(f"results_{model_name.lower()}")
    )
    mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
    (mteb_save_path).mkdir(exist_ok=True, parents=True)

    dict_results = MTEBEval(
        wrapped_model=wrapped_model,
        tasks=tasks,
        path_to_save=mteb_save_path,
        eval_rep="av",
        layer_number=layer_number,
    )
    [eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number 0
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/.pyenv/versions/miniconda3-latest/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()
/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/skle

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

- Banking77Classification, s2s

...

STS

- STS15, s2s

- STS16, s2s

- STSBenchmark, s2s

Clustering:   0%|                                                                                                                                                                                         | 0/31 [00:00<?, ?it/s]/.pyenv/versions/miniconda3-latest/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can eit

# Sanity check
TODOS:
- 1. evaluation per layers but only layer 12 and only the tasks that don't match for each of the models
- 2. Run the same thing but with the full models in the old codebase
- 3. Run the same thing but with the full models in the new codebase

## Fine-tuned MPNet (Crops)

In [ ]:
tasks = [
    # "BiorxivClusteringP2P",
    # "MedrxivClusteringP2P",
    # "RedditClusteringP2P",
    # "StackExchangeClusteringP2P",
    # "ArguAna",
    # "AmazonPolarityClassification",
    "Banking77Classification",
    # "ImdbClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    # "MTOPDomainClassification",
]

### (1)

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "crops"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

# wrap model
wrapped_model = FineTunedHFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda
CPU times: user 171 ms, sys: 165 ms, total: 336 ms
Wall time: 3.22 s


In [ ]:
# eval
eval_results = defaultdict(list)


layer_number = 12
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}_{finetune_type}_finetuning")
    / Path("sanity_check")
)
mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=layer_number,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)
df_eval_results

layer number 12


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

- MassiveIntentClassification, s2s, multilingual 51 / 51 Subsets

- MassiveScenarioClassification, s2s, multilingual 51 / 51 Subsets

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

In [ ]:
df_eval_results

,layer,Banking77Classification,MassiveIntentClassification,MassiveScenarioClassification
0,12,0.758961,0.435407,0.460323


### (2)

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "crops"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

# wrap model
wrapped_model = FineTunedHFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda
CPU times: user 86.3 ms, sys: 6.5 ms, total: 92.8 ms
Wall time: 220 ms


In [ ]:
# eval
eval_results = defaultdict(list)


layer_number = "None"
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}_{finetune_type}_finetuning")
    / Path("sanity_check")
)
mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=None,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)
df_eval_results

layer number None


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

- MassiveIntentClassification, s2s, multilingual 51 / 51 Subsets

- MassiveScenarioClassification, s2s, multilingual 51 / 51 Subsets

### (3)

In [ ]:
tasks = [
    "ArxivClusteringP2P",
    "BiorxivClusteringP2P",
    "MedrxivClusteringP2P",
    "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    "SciDocsRR",
    "MindSmallReranking",
    "SCIDOCS",
    "ArguAna",
    "STS15",
    "STS16",
    "STSBenchmark",
    ##classification
    "AmazonPolarityClassification",  # sentiment: binary positive or negative
    "Banking77Classification",
    "ImdbClassification",  # sentiment: binary positive or negative
    "MassiveIntentClassification",  # multi
    "MassiveScenarioClassification",  # multi
    "MTOPDomainClassification",  # multi
    "TweetSentimentExtractionClassification",  # sentiment: positive, neutral negative
]

In [ ]:
tasks_mteb = mteb.get_tasks(tasks=tasks)
evaluation = mteb.MTEB(tasks=tasks_mteb, langs=["en"])
# TODO: comment out english only and rerun as setting (4) if still no answer to sanity check

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "crops"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
# Load your custom base model and tokenizer
custom_base_model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

custom_tokenizer = AutoTokenizer.from_pretrained(model_path)

# Create a new SentenceTransformer model
new_modules = []

# Wrap your custom base model in a Transformer module
transformer_model = models.Transformer(
    model_name_or_path="bert-base-uncased",  # None gives an error,
    # so I initialize another model that
    # will be substituted by my custom model below
    max_seq_length=384,  # You can adjust this as needed
    do_lower_case=False,  # Adjust based on your tokenizer
)
# Replace the auto_model in the Transformer wrapper with your custom model
transformer_model.auto_model = custom_base_model
# Set the tokenizer
transformer_model.tokenizer = custom_tokenizer

# Add the wrapped model as the first module
new_modules.append(transformer_model)

# Add Pooling layer
pooling_model = models.Pooling(
    word_embedding_dimension=custom_base_model.config.hidden_size,
    pooling_mode_cls_token=False,
    pooling_mode_mean_tokens=True,
    pooling_mode_max_tokens=False,
    pooling_mode_mean_sqrt_len_tokens=False,
)
new_modules.append(pooling_model)

# Add Normalize layer
new_modules.append(models.Normalize())

# Create the new SentenceTransformer model
finetuned_mpnet = SentenceTransformer(modules=new_modules)

In [ ]:
saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("sanity_check")
)
(variables_path / saving_path).mkdir(exist_ok=True)

results = evaluation.run(
    finetuned_mpnet,
    output_folder=variables_path
    / saving_path
    / f"results_{model_name.lower()}_{finetune_type}_finetuning",
)

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

- Banking77Classification, s2s

- ImdbClassification, p2p

...

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

### (4)

In [ ]:
tasks = [
    "ArxivClusteringP2P",
    "BiorxivClusteringP2P",
    "MedrxivClusteringP2P",
    "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    "SciDocsRR",
    "MindSmallReranking",
    "SCIDOCS",
    "ArguAna",
    "STS15",
    "STS16",
    "STSBenchmark",
    ##classification
    "AmazonPolarityClassification",  # sentiment: binary positive or negative
    "Banking77Classification",
    "ImdbClassification",  # sentiment: binary positive or negative
    "MassiveIntentClassification",  # multi
    "MassiveScenarioClassification",  # multi
    "MTOPDomainClassification",  # multi
    "TweetSentimentExtractionClassification",  # sentiment: positive, neutral negative
]

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "crops"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

# wrap model
wrapped_model = FineTunedHFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda
CPU times: user 185 ms, sys: 167 ms, total: 351 ms
Wall time: 3.19 s


In [ ]:
# eval
eval_results = defaultdict(list)


layer_number = 12
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}_{finetune_type}_finetuning")
    / Path("sanity_check_v2")
)
mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=layer_number,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)
df_eval_results

layer number 12


av


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

...

- STS15, s2s

- STS16, s2s

- STSBenchmark, s2s

Clustering: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [11:01<00:00, 66.20s/it]
Repo card metadata block was not found. Setting CardData to empty.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism

,layer,ArxivClusteringP2P,BiorxivClusteringP2P,MedrxivClusteringP2P,RedditClusteringP2P,StackExchangeClusteringP2P,SciDocsRR,MindSmallReranking,SCIDOCS,ArguAna,STS15,STS16,STSBenchmark,AmazonPolarityClassification,Banking77Classification,ImdbClassification,MassiveIntentClassification,MassiveScenarioClassification,MTOPDomainClassification,TweetSentimentExtractionClassification
0,12,0.381873,0.323375,0.304291,0.553761,0.311792,0.736115,0.301656,0.12999,0.50477,0.724795,0.759524,0.717021,0.624589,0.677435,0.648364,0.335205,0.380632,0.851231,0.496746


### (5)

In [ ]:
tasks = [
    # "ArxivClusteringP2P",
    # "BiorxivClusteringP2P",
    # "MedrxivClusteringP2P",
    # "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    # "SciDocsRR",
    "MindSmallReranking",
    # "SCIDOCS",
    "ArguAna",
    # "STS15",
    "STS16",
    # "STSBenchmark",
    ##classification
    # "AmazonPolarityClassification",  # sentiment: binary positive or negative
    # "Banking77Classification",
    "ImdbClassification",  # sentiment: binary positive or negative
    # "MassiveIntentClassification",  # multi
    # "MassiveScenarioClassification",  # multi
    # "MTOPDomainClassification",  # multi
    # "TweetSentimentExtractionClassification",  # sentiment: positive, neutral negative
]

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "crops"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

# wrap model
wrapped_model = FineTunedHFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda
CPU times: user 127 ms, sys: 20.2 ms, total: 147 ms
Wall time: 2.92 s


In [ ]:
# eval
eval_results = defaultdict(list)


layer_number = 12
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}_{finetune_type}_finetuning")
    / Path("sanity_check_v3")
)
mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=layer_number,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)
df_eval_results

layer number 12


av


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- ImdbClassification, p2p

...

- ArguAna, s2p

STS

- STS16, s2s

Clustering: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [11:00<00:00, 66.01s/it]
Repo card metadata block was not found. Setting CardData to empty.
/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  pid = os.fork()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling

,layer,StackExchangeClusteringP2P,MindSmallReranking,ArguAna,STS16,ImdbClassification
0,12,0.311792,0.301656,0.50477,0.759524,0.648364


In [ ]:
df_eval_results.T * 100

,0
layer,1200.000000
StackExchangeClusteringP2P,31.179175
MindSmallReranking,30.165636
ArguAna,50.477000
STS16,75.952438
ImdbClassification,64.836400


## Fine-tuned MPNet (Dropout)

In [ ]:
tasks = [
    # "BiorxivClusteringP2P",
    # "MedrxivClusteringP2P",
    # "RedditClusteringP2P",
    # "StackExchangeClusteringP2P",
    # "ArguAna",
    # "AmazonPolarityClassification",
    "Banking77Classification",
    # "ImdbClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    # "MTOPDomainClassification",
]

### (1)

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "simcse"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

# wrap model
wrapped_model = FineTunedHFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda
CPU times: user 218 ms, sys: 190 ms, total: 409 ms
Wall time: 2.64 s


In [ ]:
# eval
eval_results = defaultdict(list)

layer_number =12
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}_{finetune_type}_finetuning")
    / Path("sanity_check")
)
mteb_save_path = (
    variables_path / saving_path / Path(f"layer_{layer_number}")
)
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=layer_number,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number 6


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

- Banking77Classification, s2s

...

STS

- STS15, s2s

- STS16, s2s

- STSBenchmark, s2s

Clustering:   0%|                                                                                                                                                                                         | 0/10 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Clustering:  10%|█████████████████▌                                                                                            

### (2)

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "simcse"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

# wrap model
wrapped_model = FineTunedHFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda
CPU times: user 218 ms, sys: 190 ms, total: 409 ms
Wall time: 2.64 s


In [ ]:
# eval
eval_results = defaultdict(list)

layer_number ="None"
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}_{finetune_type}_finetuning")
    / Path("sanity_check")
)
mteb_save_path = (
    variables_path / saving_path / Path(f"layer_{layer_number}")
)
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=None,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number 6


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

- Banking77Classification, s2s

...

STS

- STS15, s2s

- STS16, s2s

- STSBenchmark, s2s

Clustering:   0%|                                                                                                                                                                                         | 0/10 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Clustering:  10%|█████████████████▌                                                                                            

### (3)

In [ ]:
tasks = [
    "ArxivClusteringP2P",
    "BiorxivClusteringP2P",
    "MedrxivClusteringP2P",
    "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    "SciDocsRR",
    "MindSmallReranking",
    "SCIDOCS",
    "ArguAna",
    "STS15",
    "STS16",
    "STSBenchmark",
    ##classification
    "AmazonPolarityClassification",  # sentiment: binary positive or negative
    "Banking77Classification",
    "ImdbClassification",  # sentiment: binary positive or negative
    "MassiveIntentClassification",  # multi
    "MassiveScenarioClassification",  # multi
    "MTOPDomainClassification",  # multi
    "TweetSentimentExtractionClassification",  # sentiment: positive, neutral negative
]

In [ ]:
tasks_mteb = mteb.get_tasks(tasks=tasks)
evaluation = mteb.MTEB(tasks=tasks_mteb, langs=["en"])
# TODO: comment out english only and rerun as setting (4) if still no answer to sanity check

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "simcse"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
# Load your custom base model and tokenizer
custom_base_model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

custom_tokenizer = AutoTokenizer.from_pretrained(model_path)

# Create a new SentenceTransformer model
new_modules = []

# Wrap your custom base model in a Transformer module
transformer_model = models.Transformer(
    model_name_or_path="bert-base-uncased",  # None gives an error,
    # so I initialize another model that
    # will be substituted by my custom model below
    max_seq_length=384,  # You can adjust this as needed
    do_lower_case=False,  # Adjust based on your tokenizer
)
# Replace the auto_model in the Transformer wrapper with your custom model
transformer_model.auto_model = custom_base_model
# Set the tokenizer
transformer_model.tokenizer = custom_tokenizer

# Add the wrapped model as the first module
new_modules.append(transformer_model)

# Add Pooling layer
pooling_model = models.Pooling(
    word_embedding_dimension=custom_base_model.config.hidden_size,
    pooling_mode_cls_token=False,
    pooling_mode_mean_tokens=True,
    pooling_mode_max_tokens=False,
    pooling_mode_mean_sqrt_len_tokens=False,
)
new_modules.append(pooling_model)

# Add Normalize layer
new_modules.append(models.Normalize())

# Create the new SentenceTransformer model
finetuned_mpnet = SentenceTransformer(modules=new_modules)

In [ ]:
saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("sanity_check")
)
(variables_path / saving_path).mkdir(exist_ok=True)

results = evaluation.run(
    finetuned_mpnet,
    output_folder=variables_path
    / saving_path
    / f"results_{model_name.lower()}_{finetune_type}_finetuning",
)

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

- Banking77Classification, s2s

- ImdbClassification, p2p

...

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

### (4)

In [ ]:
tasks = [
    "ArxivClusteringP2P",
    "BiorxivClusteringP2P",
    "MedrxivClusteringP2P",
    "RedditClusteringP2P",
    "StackExchangeClusteringP2P",
    "SciDocsRR",
    "MindSmallReranking",
    "SCIDOCS",
    "ArguAna",
    "STS15",
    "STS16",
    "STSBenchmark",
    ##classification
    "AmazonPolarityClassification",  # sentiment: binary positive or negative
    "Banking77Classification",
    "ImdbClassification",  # sentiment: binary positive or negative
    "MassiveIntentClassification",  # multi
    "MassiveScenarioClassification",  # multi
    "MTOPDomainClassification",  # multi
    "TweetSentimentExtractionClassification",  # sentiment: positive, neutral negative
]

In [ ]:
model_name = "MPNet"
model_path = "microsoft/mpnet-base"
finetune_type = "simcse"

checkpoint_dir = Path(
    f"updated_dataset/model_checkpoints/{model_name.lower()}_{finetune_type}_finetuning"
)

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(variables_path / checkpoint_dir)

# wrap model
wrapped_model = FineTunedHFModelWrapper(model, tokenizer)

Model:  MPNet
Running on device: cuda
CPU times: user 128 ms, sys: 27.6 ms, total: 156 ms
Wall time: 2.8 s


In [ ]:
# eval
eval_results = defaultdict(list)


layer_number = 12
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}_{finetune_type}_finetuning")
    / Path("sanity_check_v2")
)
mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=layer_number,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)
df_eval_results

layer number 12


av


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- AmazonPolarityClassification, p2p

...

STS

- STS15, s2s

- STS16, s2s

- STSBenchmark, s2s

Clustering:   0%|                                                                                                                                                                      | 0/31 [00:00<?, ?it/s]

## Pre-trained SBERT

In [ ]:
tasks = [
    # "BiorxivClusteringP2P",
    # "MedrxivClusteringP2P",
    # "RedditClusteringP2P",
    # "StackExchangeClusteringP2P",
    # "ArguAna",
    # "AmazonPolarityClassification",
    "Banking77Classification",
    # "ImdbClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    # "MTOPDomainClassification",
]

### (1)

In [ ]:
model_name = "SBERT"
model_path = "sentence-transformers/all-mpnet-base-v2"

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)

# wrap model
wrapped_model = HFModelWrapper(model, tokenizer)

Model:  SBERT
Running on device: cuda
CPU times: user 127 ms, sys: 26.6 ms, total: 153 ms
Wall time: 3.98 s


In [ ]:
# eval
eval_results = defaultdict(list)

layer_number = 12
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}")
    / Path("sanity_check")
)
mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=layer_number,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number 12


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

- MassiveIntentClassification, s2s, multilingual 51 / 51 Subsets

- MassiveScenarioClassification, s2s, multilingual 51 / 51 Subsets

/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  pid = os.fork()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, 

KeyboardInterrupt: 

Traceback (most recent call last):
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py", line 426, in _process_worker
    call_item = call_queue.get(block=True, timeout=timeout)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/multiprocessing/queues.py", line 108, in get
    if not self._rlock.acquire(block, timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt



### (2)

In [ ]:
model_name = "SBERT"
model_path = "sentence-transformers/all-mpnet-base-v2"

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)

# wrap model
wrapped_model = HFModelWrapper(model, tokenizer)

Model:  SBERT
Running on device: cuda
CPU times: user 90.7 ms, sys: 9.72 ms, total: 100 ms
Wall time: 336 ms


In [ ]:
# eval
eval_results = defaultdict(list)

layer_number = "None"
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}")
    / Path("sanity_check")
)
mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=None,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number None


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

- MassiveIntentClassification, s2s, multilingual 51 / 51 Subsets

- MassiveScenarioClassification, s2s, multilingual 51 / 51 Subsets

/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  pid = os.fork()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, 

### (3)

In [ ]:
tasks_mteb = mteb.get_tasks(tasks=tasks)
evaluation = mteb.MTEB(tasks=tasks_mteb, langs=["en"])
# TODO: comment out english only and rerun as setting (4) if still no answer to sanity check

In [ ]:
model_name = "SBERT"
model_path = "sentence-transformers/all-mpnet-base-v2"

In [ ]:
pretrained_sbert = SentenceTransformer(model_path)

In [ ]:
saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("sanity_check")
)
(variables_path / saving_path).mkdir(exist_ok=True)

results = evaluation.run(
    pretrained_sbert,
    output_folder=variables_path
    / saving_path
    / f"results_{model_name.lower()}",
)

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

- MassiveIntentClassification, s2s, multilingual 51 / 51 Subsets

- MassiveScenarioClassification, s2s, multilingual 51 / 51 Subsets

### (4)

In [ ]:
tasks = [
    # "BiorxivClusteringP2P",
    # "MedrxivClusteringP2P",
    # "RedditClusteringP2P",
    # "StackExchangeClusteringP2P",
    # "ArguAna",
    # "AmazonPolarityClassification",
    "Banking77Classification",
    # "ImdbClassification",
    "MassiveIntentClassification",
    "MassiveScenarioClassification",
    # "MTOPDomainClassification",
]

In [ ]:
model_name = "SBERT"
model_path = "sentence-transformers/all-mpnet-base-v2"

In [ ]:
%%time

# fix random seeds
fix_all_seeds()

# set up model
print("Model: ", model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on device: {}".format(device))

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)

# wrap model
wrapped_model = HFModelWrapper(model, tokenizer)

Model:  SBERT
Running on device: cuda
CPU times: user 90.7 ms, sys: 9 ms, total: 99.7 ms
Wall time: 337 ms


In [ ]:
# eval
eval_results = defaultdict(list)

layer_number = 12
print("layer number", layer_number)
eval_results["layer"].append(layer_number)

saving_path = (
    Path("embeddings_" + model_name.lower())
    / Path("updated_dataset")
    / Path("mteb_benchmark")
    / Path("eval_layers")
    / Path(f"results_{model_name.lower()}")
    / Path("sanity_check_v2")
)
mteb_save_path = variables_path / saving_path / Path(f"layer_{layer_number}")
(mteb_save_path).mkdir(exist_ok=True, parents=True)

dict_results = MTEBEval(
    wrapped_model=wrapped_model,
    tasks=tasks,
    path_to_save=mteb_save_path,
    eval_rep="av",
    layer_number=layer_number,
)
[eval_results[k].append(v) for k, v in dict_results.items()]


# save
df_eval_results = pd.DataFrame(eval_results)
df_eval_results.to_parquet(
    variables_path / saving_path / "df_eval_results",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

layer number 12
av


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

- MassiveIntentClassification, s2s, multilingual 51 / 51 Subsets

- MassiveScenarioClassification, s2s, multilingual 51 / 51 Subsets